# AI天気予報

## 気象庁から漁業気象通報放送の情報を取得する
以下の気象庁のサイトから各地の気象情報をダウンロードする。
まずはgoogle driveをマウントします。


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## データを保存するフォルダを作る


## クラウド実行用のスクレイピングコード

１週間分のデータを取得するスクリプト

ニューラルネットに入力するフォーマットにデータを修正

In [2]:
import datetime
import urllib.request
import os
import json
import numpy as np
from google.colab import drive

# 1. Googleドライブをマウント（未接続の場合のみ最初の # を消して実行）
# drive.mount('/content/drive')

# ベースとなる保存先フォルダ
base_folder = '/content/drive/MyDrive/weather_data_root'

# 全角英数字を半角に直す辞書
ZEN_TO_HAN = str.maketrans("０１２３４５６７８９", "0123456789")

# 風向を「角度（度数法）」に変換するための辞書マスター
DIRECTION_TO_DEGREE = {
    "北": 0.0, "北北東": 22.5, "北東": 45.0, "東北東": 67.5,
    "東": 90.0, "東南東": 112.5, "南東": 135.0, "南南東": 157.5,
    "南": 180.0, "南南西": 202.5, "南西": 225.0, "西南西": 247.5,
    "西": 270.0, "西北西": 292.5, "北西": 315.0, "北北西": 337.5,
    "無風": -1.0
}

# ⭐ 天気文字列を「AI用通し番号」に変換するための辞書マスター
# 晴れに近いものから順に番号を割り振っています
WEATHER_TO_CODE = {
    "快晴": 0, "晴れ": 1, "薄曇り": 2, "うすぐもり": 2, "くもり": 3,
    "煙霧": 4, "砂塵嵐": 5, "地吹雪": 6, "霧": 7, "霧雨": 8,
    "雨": 9, "大雨": 10, "みぞれ": 11, "雪": 12, "大雪": 13,
    "あられ": 14, "ひょう": 15, "雷": 16, "入電なし": -1, "不明": -1
}

# 角度をsinとcosに変換する関数
def calc_sin_cos(degree):
    if degree == -1.0:
        return 0.0, 0.0
    radians = np.radians(degree)
    return float(np.sin(radians)), float(np.cos(radians))

print("========== 天気マッピング搭載パイプライン 起動 ==========")

# 1日前〜6日前（range(1, 7)）の安全ループ
for day_ago in range(1, 7):

    target_date = datetime.date.today() - datetime.timedelta(days=day_ago)
    year_str  = target_date.strftime("%Y")
    month_str = target_date.strftime("%m")
    date_str  = target_date.strftime("%Y%m%d")

    target_dir = os.path.join(base_folder, year_str, f"{year_str}-{month_str}")
    os.makedirs(target_dir, exist_ok=True)

    output_file = os.path.join(target_dir, f"ai_input_{date_str}.json")

    print(f"\n[STEP 1] 日付: {date_str} (day{day_ago}) の解析...")

    # 既存ファイルはスキップ（最新仕様で再生成したい場合は、ドライブの既存ファイルを消してから実行してください）
    if os.path.exists(output_file):
        print(f"  -> 【確認】「ai_input_{date_str}.json」は取得済みのためスキップします。")
        continue

    url = f"https://www.data.jma.go.jp/yoho/gyogyou/day{day_ago}_12.txt"
    print(f"  -> 【新規取得】気象庁からデータをダウンロード: {url}")

    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            raw_text = response.read().decode("shift_jis")

        ai_ready_data = {
            "date": date_str,
            "observation_time": "12:00",
            "locations": []
        }

        in_weather_section = False
        for line in raw_text.splitlines():
            line = line.strip()

            if "各地の天気" in line:
                in_weather_section = True
                continue
            if "船舶からの報告" in line or "漁業気象" in line:
                in_weather_section = False

            if in_weather_section and line:
                if "風力" in line:
                    left_side, right_side = line.split("風力", 1)
                    left_parts = left_side.split()
                    if not left_parts:
                        continue

                    wind_direction = left_parts[-1]
                    place_name = "".join(left_parts[:-1])

                    right_parts = right_side.split()
                    if len(right_parts) < 3:
                        continue

                    # 1. 風力数値の抽出
                    wind_power_str = right_parts[0].translate(ZEN_TO_HAN)
                    wind_power = int(wind_power_str) if wind_power_str.isdigit() else 0

                    # 2. 天気文字列の抽出
                    weather_word = right_parts[1]

                    # 3. 気圧数値のクレンジングと4桁復元
                    press_word = right_parts[2]
                    clean_press = press_word.replace("hPa", "").replace("ｈＰａ", "").replace("微圧計", "").strip()
                    press_digits_str = clean_press.translate(ZEN_TO_HAN)
                    try:
                        pressure_2digits = int(press_digits_str)
                        if pressure_2digits >= 85:
                            pressure_hpa = 900 + pressure_2digits
                        else:
                            pressure_hpa = 1000 + pressure_2digits
                    except ValueError:
                        pressure_hpa = None

                    # 4. 気温数値の抽出
                    temperature_c = None
                    if len(right_parts) > 3:
                        temp_word = right_parts[3]
                        clean_temp = temp_word.replace("度", "").strip()
                        temp_digits_str = clean_temp.translate(ZEN_TO_HAN)
                        try:
                            temperature_c = int(temp_digits_str)
                        except ValueError:
                            temperature_c = None

                    # 5. 風向のsin/cos変換
                    degree = DIRECTION_TO_DEGREE.get(wind_direction, -1.0)
                    wind_sin, wind_cos = calc_sin_cos(degree)

                    # ⭐【進化ポイント】天気の文字列を数値（コード）に変換
                    # もしマスターにない特殊な天気やノイズがあれば、安全のため「99（その他）」に割り振る
                    weather_code = WEATHER_TO_CODE.get(weather_word, 99)

                    ai_ready_data["locations"].append({
                        "place": place_name,
                        "wind_direction": wind_direction,
                        "wind_degree": degree,
                        "wind_sin": wind_sin,
                        "wind_cos": wind_cos,
                        "wind_power": wind_power,
                        "weather": weather_word,
                        "weather_code": weather_code,  # AI用：天気の数値コード
                        "pressure_hpa": pressure_hpa,
                        "temperature_c": temperature_c
                    })

        # 保存
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(ai_ready_data, f, ensure_ascii=False, indent=4)

        print(f"  -> 【成功】「ai_input_{date_str}.json」に天気コードを組み込んで保存しました。")

    except Exception as e:
        print(f"  -> 【エラー】処理中に問題が発生しました: {e}")

print("\n========== すべてのデータが完全なAI仕様になりました ==========")

========== 天気マッピング搭載パイプライン 起動 ==========

[STEP 1] 日付: 20260823 (day1) の解析...
  -> 【新規取得】気象庁からデータをダウンロード: https://www.data.jma.go.jp/yoho/gyogyou/day1_12.txt
  -> 【成功】「ai_input_20260823.json」に天気コードを組み込んで保存しました。

[STEP 1] 日付: 20260822 (day2) の解析...
  -> 【新規取得】気象庁からデータをダウンロード: https://www.data.jma.go.jp/yoho/gyogyou/day2_12.txt
  -> 【成功】「ai_input_20260822.json」に天気コードを組み込んで保存しました。

[STEP 1] 日付: 20260821 (day3) の解析...
  -> 【新規取得】気象庁からデータをダウンロード: https://www.data.jma.go.jp/yoho/gyogyou/day3_12.txt
  -> 【成功】「ai_input_20260821.json」に天気コードを組み込んで保存しました。

[STEP 1] 日付: 20260820 (day4) の解析...
  -> 【確認】「ai_input_20260820.json」は取得済みのためスキップします。

[STEP 1] 日付: 20260819 (day5) の解析...
  -> 【確認】「ai_input_20260819.json」は取得済みのためスキップします。

[STEP 1] 日付: 20260818 (day6) の解析...
  -> 【確認】「ai_input_20260818.json」は取得済みのためスキップします。

========== すべてのデータが完全なAI仕様になりました ==========


In [3]:
import os
import json
import pandas as pd
from google.colab import drive

# 1. Googleドライブをマウント（未接続の場合のみ実行）
# drive.mount('/content/drive')

# 探索を開始する一番上のフォルダ
base_folder = '/content/drive/MyDrive/weather_data_root'

file_registry = {}

print("--- Googleドライブ内を総当たりでスキャン中... ---")

# ✨【修正ポイント】os.walk を使い、年/月フォルダの形式がどうであれ、
# weather_data_root の中にある「すべてのフォルダ」を自動でくまなく探します
for root, dirs, files in os.walk(base_folder):
    for filename in files:
        # ファイル名が「ai_input_」で始まり「.json」で終わるものだけを抽出
        if filename.startswith("ai_input_") and filename.endswith(".json"):
            # ファイル名から日付の8桁数字を抜き出す
            date_part = filename.replace("ai_input_", "").replace(".json", "")

            if date_part.isdigit() and len(date_part) == 8:
                # フルパスを辞書に登録
                filepath = os.path.join(root, filename)
                file_registry[date_part] = filepath

print(f"--- スキャン完了: 計 {len(file_registry)} 件の気象ファイルを発見しました ---")
if len(file_registry) > 0:
    print("見つかった日付の例:", sorted(list(file_registry.keys())))

dataset_rows = []

# ヒントとして使いたい主要な観測地点（部分一致）
FEATURE_PLACES = ["石垣", "那覇", "鹿児島", "潮岬", "御前崎", "前橋", "八丈島", "輪島", "仙台", "札幌", "根室"]

# 2. 登録された日付をループして、翌日のファイルが存在するペアを自動結合
for t_str, today_file in sorted(file_registry.items()):
    current_date = pd.to_datetime(t_str)
    tomorrow_date = current_date + pd.Timedelta(days=1)
    tomorrow_str = tomorrow_date.strftime("%Y%m%d")

    # 翌日のファイルがスキャン結果に存在しなければスキップ
    if tomorrow_str not in file_registry:
        continue

    tomorrow_file = file_registry[tomorrow_str]

    try:
        # 3. ファイルの読み込み
        with open(today_file, "r", encoding="utf-8") as f:
            today_data = json.load(f)
        with open(tomorrow_file, "r", encoding="utf-8") as f:
            tomorrow_data = json.load(f)

        # 入力データの作成
        row_data = {"date": t_str}

        for loc in today_data["locations"]:
            p_name = loc["place"]
            for target_place in FEATURE_PLACES:
                if target_place in p_name:
                    row_data[f"{target_place}_pressure"] = loc["pressure_hpa"]
                    row_data[f"{target_place}_weather"] = loc["weather_code"]
                    # Add wind_sin and wind_cos
                    row_data[f"{target_place}_wind_sin"] = loc["wind_sin"]
                    row_data[f"{target_place}_wind_cos"] = loc["wind_cos"]
                    break

        # 正解ラベル（翌日の前橋の天気）の作成
        tomorrow_omaezaki_weather = None
        for loc in tomorrow_data["locations"]:
            if "前橋" in loc["place"]:
                tomorrow_omaezaki_weather = loc["weather_code"]
                break

        # ペアが揃ったのでデータセットの1行として追加
        if tomorrow_omaezaki_weather is not None:
            row_data["TARGET_tomorrow_omaezaki_weather"] = tomorrow_omaezaki_weather
            dataset_rows.append(row_data)

    except Exception as e:
        # 万が一壊れたファイルがあった場合の安全策
        print(f"【警告】ファイル {t_str} の処理中にエラー（スキップします）: {e}")

# 二次元の表（データフレーム）に一括変換
df_dataset = pd.DataFrame(dataset_rows)

print(f"\n【結合完了】ペアが揃ったAI学習用データセットが {len(df_dataset)} 日分作成されました！")
if len(df_dataset) > 0:
    print("\n--- 作成されたデータセットの見た目（最初の数行） ---")
    print(df_dataset.head())

df_dataset.head()

--- Googleドライブ内を総当たりでスキャン中... ---
--- スキャン完了: 計 33 件の気象ファイルを発見しました ---
見つかった日付の例: ['20260720', '20260721', '20260722', '20260723', '20260724', '20260725', '20260727', '20260728', '20260729', '20260730', '20260731', '20260801', '20260802', '20260804', '20260805', '20260806', '20260807', '20260808', '20260809', '20260810', '20260811', '20260812', '20260813', '20260814', '20260815', '20260816', '20260817', '20260818', '20260819', '20260820', '20260821', '20260822', '20260823']

【結合完了】ペアが揃ったAI学習用データセットが 30 日分作成されました！

--- 作成されたデータセットの見た目（最初の数行） ---
       date  石垣_pressure  石垣_weather  石垣_wind_sin  石垣_wind_cos  那覇_pressure  \
0  20260720         1009           1    -0.707107    -0.707107         1009   
1  20260721         1010           1    -0.382683    -0.923880         1011   
2  20260722         1010           1    -0.382683    -0.923880         1010   
3  20260723         1009           1     0.382683    -0.923880         1010   
4  20260724         1009           1     0.923880    -

,date,石垣_pressure,石垣_weather,石垣_wind_sin,石垣_wind_cos,那覇_pressure,那覇_weather,那覇_wind_sin,那覇_wind_cos,鹿児島_pressure,...,輪島_wind_cos,仙台_pressure,仙台_weather,仙台_wind_sin,仙台_wind_cos,根室_pressure,根室_weather,根室_wind_sin,根室_wind_cos,TARGET_tomorrow_omaezaki_weather
0,20260720,1009,1,-0.707107,-0.707107,1009,1,-0.382683,-9.238795e-01,1009,...,-0.707107,1007,1,0.707107,-0.707107,1005,3,1.224647e-16,-1.000000,1
1,20260721,1010,1,-0.382683,-0.923880,1011,1,-1.000000,-1.836970e-16,1010,...,0.923880,1008,3,0.923880,-0.382683,1006,1,3.826834e-01,-0.923880,1
2,20260722,1010,1,-0.382683,-0.923880,1010,1,-1.000000,-1.836970e-16,1010,...,-0.923880,1006,3,0.923880,-0.382683,1006,3,0.000000e+00,1.000000,1
3,20260723,1009,1,0.382683,-0.923880,1010,1,1.000000,6.123234e-17,1010,...,-0.707107,1008,3,0.923880,-0.382683,1009,1,-3.826834e-01,0.923880,1
4,20260724,1009,1,0.923880,-0.382683,1011,3,0.707107,-7.071068e-01,1012,...,-0.707107,1012,9,0.707107,-0.707107,1015,3,7.071068e-01,-0.707107,1


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("========== AI天気予報モデル（ランダムフォレスト）起動 ==========")

# 1. データの準備（df_dataset を使用）
# 欠損（NaN / None）がある行を自動で削除します
df_ml = df_dataset.dropna().copy()

# 2. 「入力（X）」と「正解（y）」に切り分ける
# 'date' と 'TARGET_tomorrow_omaezaki_weather' は特徴量から除外
X = df_ml.drop(columns=["date", "TARGET_tomorrow_omaezaki_weather"], errors="ignore")
y = df_ml["TARGET_tomorrow_omaezaki_weather"]

print(f"\n[STEP 1] データの切り分け完了")
print(f"  -> 入力データの項目数（特徴量の数）: {X.shape[1]} 項目")
print(f"  -> 学習させる総日数: {len(df_ml)} 日分")

# 3. データの学習（初期データが少ない間はすべてのデータを脳みそに覚え込ませます）
X_train, y_train = X, y

# 4. AIモデル（ランダムフォレスト）の作成と学習
model = RandomForestClassifier(n_estimators=100, random_state=42)

print("\n[STEP 2] AIに気象配置データを学習中（Fit）...")
model.fit(X_train, y_train)
print("  -> 学習が正常に完了しました！AIが気圧配置のパターンを記憶しました。")

# 5. 覚えたデータに対する「予測テスト」
y_pred = model.predict(X_train)

# 6. 予測精度の計算（正解率）
accuracy = accuracy_score(y_train, y_pred)
print(f"\n[STEP 3] 学習データに対するAI予報の正解率: {accuracy * 100:.1f} %")

# 7. 明日の予報をシミュレーションしてみる
print("\n[STEP 4] 最新の気圧配置から「明日の前橋の天気」を予測！")
latest_features = X.iloc[[-1]] # 一番最後の行（最新日）を抽出
predicted_code = model.predict(latest_features)[0]

# 予測された確率（自信度）をパーセントで出す
probabilities = model.predict_proba(latest_features)[0]
max_prob = np.max(probabilities)

# 数値コードを人間の言葉に戻す対応表
CODE_TO_WEATHER = {0: "快晴", 1: "晴れ", 2: "薄曇り", 3: "くもり", 4: "煙霧", 7: "霧", 9: "雨", 10: "大雨", 12: "雪"}
predicted_weather = CODE_TO_WEATHER.get(predicted_code, f"コード {predicted_code} の天気")

print(f"  -> AIの予測結果: 【 {predicted_weather} 】")
print(f"  -> AIの予測の自信度: {max_prob * 100:.1f} %")

========== AI天気予報モデル（ランダムフォレスト）起動 ==========

[STEP 1] データの切り分け完了
  -> 入力データの項目数（特徴量の数）: 40 項目
  -> 学習させる総日数: 29 日分

[STEP 2] AIに気象配置データを学習中（Fit）...
  -> 学習が正常に完了しました！AIが気圧配置のパターンを記憶しました。

[STEP 3] 学習データに対するAI予報の正解率: 96.6 %

[STEP 4] 最新の気圧配置から「明日の前橋の天気」を予測！
  -> AIの予測結果: 【 くもり 】
  -> AIの予測の自信度: 71.0 %


In [5]:
import os
import json
import pandas as pd
from google.colab import drive

# 1. Googleドライブをマウント（未接続の場合のみ実行）
# drive.mount('/content/drive')

# 探索を開始する一番上のフォルダ
base_folder = '/content/drive/MyDrive/weather_data_root'

file_registry = {}

print("--- Googleドライブ内を総当たりでスキャン中... ---")

# ✨【修正ポイント】os.walk を使い、年/月フォルダの形式がどうであれ、
# weather_data_root の中にある「すべてのフォルダ」を自動でくまなく探します
for root, dirs, files in os.walk(base_folder):
    for filename in files:
        # ファイル名が「ai_input_」で始まり「.json」で終わるものだけを抽出
        if filename.startswith("ai_input_") and filename.endswith(".json"):
            # ファイル名から日付の8桁数字を抜き出す
            date_part = filename.replace("ai_input_", "").replace(".json", "")

            if date_part.isdigit() and len(date_part) == 8:
                # フルパスを辞書に登録
                filepath = os.path.join(root, filename)
                file_registry[date_part] = filepath

print(f"--- スキャン完了: 計 {len(file_registry)} 件の気象ファイルを発見しました ---")
if len(file_registry) > 0:
    print("見つかった日付の例:", sorted(list(file_registry.keys())))

dataset_rows = []

# ヒントとして使いたい主要な観測地点（部分一致）
FEATURE_PLACES = ["石垣", "那覇", "鹿児島", "潮岬", "御前崎", "八丈島", "輪島", "仙台", "札幌", "根室"]

# 2. 登録された日付をループして、翌日のファイルが存在するペアを自動結合
for t_str, today_file in sorted(file_registry.items()):
    current_date = pd.to_datetime(t_str)
    tomorrow_date = current_date + pd.Timedelta(days=1)
    tomorrow_str = tomorrow_date.strftime("%Y%m%d")

    # 翌日のファイルがスキャン結果に存在しなければスキップ
    if tomorrow_str not in file_registry:
        continue

    tomorrow_file = file_registry[tomorrow_str]

    try:
        # 3. ファイルの読み込み
        with open(today_file, "r", encoding="utf-8") as f:
            today_data = json.load(f)
        with open(tomorrow_file, "r", encoding="utf-8") as f:
            tomorrow_data = json.load(f)

        # 入力データの作成
        row_data = {"date": t_str}

        for loc in today_data["locations"]:
            p_name = loc["place"]
            for target_place in FEATURE_PLACES:
                if target_place in p_name:
                    row_data[f"{target_place}_pressure"] = loc["pressure_hpa"]
                    row_data[f"{target_place}_weather"] = loc["weather_code"]
                    # Add wind_sin and wind_cos
                    row_data[f"{target_place}_wind_sin"] = loc["wind_sin"]
                    row_data[f"{target_place}_wind_cos"] = loc["wind_cos"]
                    break

        # 正解ラベル（翌日の御前崎の天気）の作成
        tomorrow_omaezaki_weather = None
        for loc in tomorrow_data["locations"]:
            if "御前崎" in loc["place"]:
                tomorrow_omaezaki_weather = loc["weather_code"]
                break

        # ペアが揃ったのでデータセットの1行として追加
        if tomorrow_omaezaki_weather is not None:
            row_data["TARGET_tomorrow_omaezaki_weather"] = tomorrow_omaezaki_weather
            dataset_rows.append(row_data)

    except Exception as e:
        # 万が一壊れたファイルがあった場合の安全策
        print(f"【警告】ファイル {t_str} の処理中にエラー（スキップします）: {e}")

# 二次元の表（データフレーム）に一括変換
df_dataset = pd.DataFrame(dataset_rows)

print(f"\n【結合完了】ペアが揃ったAI学習用データセットが {len(df_dataset)} 日分作成されました！")
if len(df_dataset) > 0:
    print("\n--- 作成されたデータセットの見た目（最初の数行） ---")
    print(df_dataset.head())

df_dataset.head()

--- Googleドライブ内を総当たりでスキャン中... ---
--- スキャン完了: 計 33 件の気象ファイルを発見しました ---
見つかった日付の例: ['20260720', '20260721', '20260722', '20260723', '20260724', '20260725', '20260727', '20260728', '20260729', '20260730', '20260731', '20260801', '20260802', '20260804', '20260805', '20260806', '20260807', '20260808', '20260809', '20260810', '20260811', '20260812', '20260813', '20260814', '20260815', '20260816', '20260817', '20260818', '20260819', '20260820', '20260821', '20260822', '20260823']

【結合完了】ペアが揃ったAI学習用データセットが 30 日分作成されました！

--- 作成されたデータセットの見た目（最初の数行） ---
       date  石垣_pressure  石垣_weather  石垣_wind_sin  石垣_wind_cos  那覇_pressure  \
0  20260720         1009           1    -0.707107    -0.707107         1009   
1  20260721         1010           1    -0.382683    -0.923880         1011   
2  20260722         1010           1    -0.382683    -0.923880         1010   
3  20260723         1009           1     0.382683    -0.923880         1010   
4  20260724         1009           1     0.923880    -

,date,石垣_pressure,石垣_weather,石垣_wind_sin,石垣_wind_cos,那覇_pressure,那覇_weather,那覇_wind_sin,那覇_wind_cos,鹿児島_pressure,...,輪島_wind_cos,仙台_pressure,仙台_weather,仙台_wind_sin,仙台_wind_cos,根室_pressure,根室_weather,根室_wind_sin,根室_wind_cos,TARGET_tomorrow_omaezaki_weather
0,20260720,1009,1,-0.707107,-0.707107,1009,1,-0.382683,-9.238795e-01,1009,...,-0.707107,1007,1,0.707107,-0.707107,1005,3,1.224647e-16,-1.000000,1
1,20260721,1010,1,-0.382683,-0.923880,1011,1,-1.000000,-1.836970e-16,1010,...,0.923880,1008,3,0.923880,-0.382683,1006,1,3.826834e-01,-0.923880,1
2,20260722,1010,1,-0.382683,-0.923880,1010,1,-1.000000,-1.836970e-16,1010,...,-0.923880,1006,3,0.923880,-0.382683,1006,3,0.000000e+00,1.000000,1
3,20260723,1009,1,0.382683,-0.923880,1010,1,1.000000,6.123234e-17,1010,...,-0.707107,1008,3,0.923880,-0.382683,1009,1,-3.826834e-01,0.923880,1
4,20260724,1009,1,0.923880,-0.382683,1011,3,0.707107,-7.071068e-01,1012,...,-0.707107,1012,9,0.707107,-0.707107,1015,3,7.071068e-01,-0.707107,1
